#### Titanic Data Analysis and JSON Export
Author: Marc Tanguy
Description: Analyze Titanic passenger data, engineer features, and export to JSON


In [13]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


In [14]:
# load the CSV into a pandas DataFrame
df = pd.read_csv(CSV_FILE)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [15]:
# `select_dtypes` returns only columns whose dtype matches the given kind.
numeric_df = df.select_dtypes(include=["number"])

# compute statistics (mean, median, std) for each numeric column
stats = pd.DataFrame({
    "Mean":   numeric_df.mean(),
    "Median": numeric_df.median(),
    "StdDev": numeric_df.std()
})

print(stats)

                   Mean    Median      StdDev
PassengerId  446.000000  446.0000  257.353842
Survived       0.383838    0.0000    0.486592
Pclass         2.308642    3.0000    0.836071
Age           29.699118   28.0000   14.526497
SibSp          0.523008    0.0000    1.102743
Parch          0.381594    0.0000    0.806057
Fare          32.204208   14.4542   49.693429


In [16]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
 
for col in df.columns:
    missing_count = missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    # Only store columns that actually have missing values
    if missing_count > 0:
        missing_data[col] = {
            "count": missing_count,
            "percent": missing_percent
        }

# Print results
for col, data in missing_data.items():
    print(f"{col}: {data['count']} missing ({data['percent']:.2f}%)")


MISSING VALUES ANALYSIS
Age: 177 missing (19.87%)
Cabin: 687 missing (77.10%)
Embarked: 2 missing (0.22%)


In [17]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1  # +1 to include the passenger themselves
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
 
# Feature 2: Is Alone
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

# Apply the function to every row in the Age column
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))
 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Family Size Analysis
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

# IsAlone Analysis
print("\nIs Alone by Survival:")
isalone_survival = df_features.groupby('Survived')['IsAlone'].agg(['mean', 'median', 'std'])
print(isalone_survival)

# AgeGroup Analysis
print("\nAge Group by Survival:")
age_survival = df_features.groupby(['Survived', 'AgeGroup']).size().unstack()
print(age_survival)

# Sex Analysis
print("\nSex by Survival:")
sex_survival = df_features.groupby(['Survived', 'Sex']).size().unstack()
print(sex_survival)

# Pclass Analysis
print("\nPassenger Class by Survival:")
pclass_survival = df_features.groupby(['Survived', 'Pclass']).size().unstack()
print(pclass_survival)

 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

print("\nIs Alone:")
print(f"  Survived mean: {survived['IsAlone'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['IsAlone'].mean():.2f}")
print(f"  Difference: {abs(survived['IsAlone'].mean() - not_survived['IsAlone'].mean()):.2f}")

print("\nAge Group:")
age_survival_counts = df_features.groupby(['Survived', 'AgeGroup']).size().unstack()
print(age_survival_counts)

print("\nSex:")
print(f"  Survived female count: {survived[survived['Sex'] == 'female'].shape[0]}")
print(f"  Survived male count: {survived[survived['Sex'] == 'male'].shape[0]}")
print(f"  Not Survived female count: {not_survived[not_survived['Sex'] == 'female'].shape[0]}")
print(f"  Not Survived male count: {not_survived[not_survived['Sex'] == 'male'].shape[0]}")

print("\nPassenger Class:")
for pclass in [1, 2, 3]:
    print(f"  Class {pclass} survived: {survived[survived['Pclass'] == pclass].shape[0]}")
    print(f"  Class {pclass} not survived: {not_survived[not_survived['Pclass'] == pclass].shape[0]}")

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

##### Insights:

Wether being single or married or the size od the family do not show a meaningful difference between survivors and not survivors, we can see the following relevant data:

-Children survived more than any other age group.

-Females survived more than males.

-Class 1 passengers are the only ones that survived more than not.

In [18]:
class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        # Initialize all passenger attributes
        # Tip: Use pd.notna() to check if a value is not null/NaN
        # Tip: Convert to appropriate types (int, float, str)
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None
        
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        # Return a dictionary with all passenger attributes
        # Tip: Dictionary keys should match the attribute names
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }
 
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        # Iterate through dataframe rows and create Passenger objects
        # Tip: Use self.dataframe.iterrows() to loop through rows
        # Tip: Use row.get('ColumnName', default_value) to safely get values
        for idx, row in self.dataframe.iterrows():
            # Create a Passenger object with data from the row
            passenger = Passenger(
                passenger_id = row.get('PassengerId', idx),
                name = row.get('Name', None),
                age = row.get('Age', None),
                sex = row.get('Sex', None),
                survived = row.get('Survived', None),
                pclass = row.get('Pclass', None),
                fare = row.get('Fare', None),
                embarked = row.get('Embarked', None),
                family_size = row.get('FamilySize', None),
                is_alone = row.get('IsAlone', None),
                title = row.get('Title', None)
            )
            self.passengers.append(passenger)
    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        # Create a dictionary with metadata and passenger data
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                # Add more metadata fields
                'total_passengers': len(self.passengers),
                'survival_rate': round(self.dataframe['Survived'].mean() * 100, 2)
            },
            'passengers': [p.to_dict() for p in self.passengers]  # Convert all passenger objects to dictionaries
        }
        
        # Write the data to a JSON file
        # Tip: Use json.dump() with indent=2 for readable formatting
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"Data exported to {filename}")
        return data
    
    def get_summary_stats(self):
        """Get summary statistics."""
        # Calculate and return summary statistics
        # Tip: Use list comprehensions to filter and calculate
        # Example for counting survived passengers:
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        
        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': len(self.passengers) - survived_count,
            # Filter out None values before calculating averages
            'average_age': round(sum(p.age for p in self.passengers if p.age is not None) / 
                          sum(1 for p in self.passengers if p.age is not None), 2),
            'average_fare': round(sum(p.fare for p in self.passengers if p.fare is not None) / 
                           sum(1 for p in self.passengers if p.fare is not None), 2)
        }
 
# Create dataset object and export
# Check if df_engineered exists and is not empty
if 'df_features' in locals() and not df_features.empty:
    # Create a TitanicDataset object
    dataset = TitanicDataset(df_features)
    
    # Rename to summary_stats to avoid conflict with stats dataframe above
    summary_stats = dataset.get_summary_stats()

    # Print basic information about the dataset
    print("Dataset Information:")
    print(f"Total passengers: {summary_stats['total_passengers']}")
    print(f"Survival rate: {dataset.dataframe['Survived'].mean() * 100:.2f}%")
    
    # Get and display summary statistics
    # Tip: Call get_summary_stats() and iterate through the results
    print("\nSummary Statistics:")
    for key, value in summary_stats.items():
        print(f"{key.replace('_', ' ').title()}: {value}")
    
    # Export to JSON (optional - uncomment when ready)
    dataset.to_json('titanic_data.json')

Dataset Information:
Total passengers: 891
Survival rate: 38.38%

Summary Statistics:
Total Passengers: 891
Survived: 342
Did Not Survive: 549
Average Age: 29.7
Average Fare: 32.2
Data exported to titanic_data.json


In [20]:

# Define the JSON file path
JSON_FILE = 'titanic_data.json'

# Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

# Print summary of JSON data and verify content
print("="*50)
print("JSON VALIDATION")
print("="*50)

# Check metadata
print("\nMetadata:")
for key, value in json_data['metadata'].items():
    print(f"  {key}: {value}")

# Check structure
print("\nJSON Keys:", list(json_data.keys()))

# Check passenger count
print(f"\nTotal passengers in JSON: {len(json_data['passengers'])}")

# Check first passenger to verify structure
print("\nFirst passenger record:")
for key, value in json_data['passengers'][0].items():
    print(f"  {key}: {value}")

# Verify all expected keys are present
expected_keys = ['passenger_id', 'name', 'age', 'sex', 'survived', 
                 'pclass', 'fare', 'embarked', 'family_size', 'is_alone', 'title']
actual_keys = list(json_data['passengers'][0].keys())
missing_keys = [k for k in expected_keys if k not in actual_keys]

print(f"\nExpected keys present: {len(missing_keys) == 0}")
if missing_keys:
    print(f"Missing keys: {missing_keys}")

# Check missing values per field
print("\nMissing values per field:")
for key in expected_keys:
    missing = sum(1 for p in json_data['passengers'] if p[key] is None)
    print(f"  {key}: {missing} missing")

# Check statistics
print("\nStatistics verification:")
print(f"  Survival rate in JSON: {json_data['metadata']['survival_rate']}%")
print(f"  Total passengers in metadata: {json_data['metadata']['total_passengers']}")
print(f"  Total passengers actual count: {len(json_data['passengers'])}")
print(f"  Counts match: {json_data['metadata']['total_passengers'] == len(json_data['passengers'])}")

JSON VALIDATION

Metadata:
  dataset_name: Titanic Passenger Dataset
  export_date: 2026-04-26T18:04:24.788272
  total_passengers: 891
  survival_rate: 38.38

JSON Keys: ['metadata', 'passengers']

Total passengers in JSON: 891

First passenger record:
  passenger_id: 1
  name: Braund, Mr. Owen Harris
  age: 22.0
  sex: male
  survived: 0
  pclass: 3
  fare: 7.25
  embarked: S
  family_size: 2
  is_alone: 0
  title: None

Expected keys present: True

Missing values per field:
  passenger_id: 0 missing
  name: 0 missing
  age: 177 missing
  sex: 0 missing
  survived: 0 missing
  pclass: 0 missing
  fare: 0 missing
  embarked: 2 missing
  family_size: 0 missing
  is_alone: 0 missing
  title: 891 missing

Statistics verification:
  Survival rate in JSON: 38.38%
  Total passengers in metadata: 891
  Total passengers actual count: 891
  Counts match: True
